# BrainTart - Validation Inference & Submission

This notebook is dedicated solely to running inference on the validation dataset using your pre-trained `best_model.pt` weights. It will:
1. Download and extract the validation data.
2. Run inference (with optional MC-Dropout and TTA).
3. Package the results and upload to Synapse.

## 1. Setup & Repo Clone

In [ ]:
import subprocess, sys, os

REPO_URL = "https://github.com/PaulOkwija/BrainTart"
REPO_DIR = "/kaggle/working/BrainTart"

if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull"])

subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "-r", os.path.join(REPO_DIR, "requirements.txt")])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "synapseclient"])
print("Dependencies ready.")
sys.path.insert(0, REPO_DIR)

## 2. Download Validation Data
Ensure you have your `brats` Kaggle Secret set up.

In [ ]:
import zipfile
from pathlib import Path
import synapseclient
from kaggle_secrets import UserSecretsClient

syn = synapseclient.Synapse()
syn.login(authToken=UserSecretsClient().get_secret("brats"))

VALIDATION_ENTITY = "syn51684975"
DOWNLOAD_DIR = Path("/kaggle/working/brats_download")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

VAL_ROOT = Path("/kaggle/working/brats_data_val")
VAL_ROOT.mkdir(parents=True, exist_ok=True)

if not any(VAL_ROOT.rglob("BraTS-GLI-*-*")):
    print("Downloading validation data...")
    entity = syn.get(entity=VALIDATION_ENTITY, version=1, downloadLocation=str(DOWNLOAD_DIR))
    
    print("Extracting (this may take a while)...")
    with zipfile.ZipFile(entity.path, "r") as zf:
        members = zf.namelist()
        from tqdm.auto import tqdm
        for member in tqdm(members, desc="Extracting to VAL_ROOT"):
            zf.extract(member, path=VAL_ROOT)
            
    candidates = list(VAL_ROOT.rglob("BraTS-GLI-*-*"))
    if candidates:
        VAL_ROOT = candidates[0].parent
else:
    candidates = list(VAL_ROOT.rglob("BraTS-GLI-*-*"))
    if candidates:
        VAL_ROOT = candidates[0].parent
    print("Validation data already extracted.")

import shutil
shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)

## 3. Run Inference
Point `CHECKPOINT_PATH` to the location of your uploaded/trained `best_model.pt`. If you uploaded weights as a Kaggle dataset, the path will look like `/kaggle/input/my-dataset/best_model.pt`.

In [ ]:
CHECKPOINT_PATH = "/kaggle/working/checkpoints/best_model.pt" # <-- UPDATE THIS PATH IF NEEDED
RESULTS_DIR = "/kaggle/working/results"

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(f"Weights not found at {CHECKPOINT_PATH}. Please provide a valid path.")

infer_cmd = [
    sys.executable, f"{REPO_DIR}/inference.py",
    "--dataset", str(VAL_ROOT),
    "--checkpoint", CHECKPOINT_PATH,
    "--results-dir", RESULTS_DIR,
    "--crop", "96", "96", "96",
    "--base-ch", "32",
    "--depth", "3",
    "--mono-stages", "3",
    "--mono-scales", "3",
    "--dropout", "0.15",  # Must match your training config
    "--mc-samples", "8",  # V2 feature: 8 stochastic passes
    "--tta"               # V2 feature: test-time augmentation flips
]

print("Running Inference...")
subprocess.run(infer_cmd, check=True)

## 4. Visualise Predictions
Let's look at a randomly selected case from the validation set to inspect the generated reconstruction.

In [ ]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import random

pred_files = list(Path(RESULTS_DIR).glob("*-t1n-inference.nii.gz"))
if pred_files:
    sample_file = random.choice(pred_files)
    case_name = sample_file.name.replace("-t1n-inference.nii.gz", "")
    
    pred_vol = nib.load(sample_file).get_fdata().astype(np.float32)
    
    # Try to load uncertainty map
    unc_file = Path(RESULTS_DIR) / "uncertainty" / f"{case_name}-uncertainty.nii.gz"
    has_unc = unc_file.exists()
    if has_unc:
        unc_vol = nib.load(unc_file).get_fdata().astype(np.float32)
        
    # Try to load voided input
    voided_matches = list(VAL_ROOT.rglob(f"{case_name}-t1n-voided.nii.gz"))
    
    n_cols = 3 if has_unc else 2
    fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5))
    
    best_z = pred_vol.shape[2] // 2
    signal_per_z = pred_vol.sum(axis=(0, 1))
    if signal_per_z.sum() > 0:
        best_z = int(signal_per_z.argmax())
        
    vmax = np.percentile(pred_vol, 99) if pred_vol.max() > 0 else 1.0
    
    col = 0
    if voided_matches:
        voided_vol = nib.load(voided_matches[0]).get_fdata().astype(np.float32)
        axes[col].imshow(voided_vol[:, :, best_z].T, cmap="gray", origin="lower", vmin=0, vmax=vmax)
        axes[col].set_title("Input (Voided)", fontsize=12)
        axes[col].axis("off")
        col += 1
    else:
        axes[col].text(0.5, 0.5, "Voided not found", ha="center", va="center")
        axes[col].axis("off")
        col += 1
        
    axes[col].imshow(pred_vol[:, :, best_z].T, cmap="gray", origin="lower", vmin=0, vmax=vmax)
    axes[col].set_title("Prediction", fontsize=12)
    axes[col].axis("off")
    col += 1
    
    if has_unc:
        im = axes[col].imshow(unc_vol[:, :, best_z].T, cmap="hot", origin="lower")
        axes[col].set_title("Uncertainty (\u03c3)", fontsize=12)
        axes[col].axis("off")
        plt.colorbar(im, ax=axes[col], fraction=0.046, pad=0.04)
        
    plt.suptitle(f"{case_name} - z={best_z}", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("No predictions found to visualise.")


## 5. Package and Upload

In [ ]:
zip_path = Path("/kaggle/working/submission.zip")
pred_files = sorted(Path(RESULTS_DIR).glob("BraTS-GLI-*-*-t1n-inference.nii.gz"))

assert len(pred_files) > 0, "No predictions generated!"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in pred_files:
        zf.write(f, arcname=f.name)

print(f"Created {zip_path.name} with {len(pred_files)} files.")

syn_up = synapseclient.Synapse()
syn_up.login(authToken=UserSecretsClient().get_secret("brats_upload"), silent=True)

SUBMISSION_FOLDER_ID = "syn75156645"
uploaded = syn_up.store(synapseclient.File(
    path=str(zip_path),
    parent=SUBMISSION_FOLDER_ID,
    description="BraTS inpainting - BrainTart Validation Inference"
))

print(f"Uploaded {uploaded.name} to {SUBMISSION_FOLDER_ID}")